* link to dataset: https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95/data_preview

In [124]:
import pandas as pd


def summarize_columns(df):
    print(
        pd.DataFrame(
            [
                (
                    c,
                    df[c].dtype,
                    len(df[c].unique()),
                    df[c].memory_usage(deep=True) // (1024**2),
                )
                for c in df.columns
            ],
            columns=["name", "dtype", "unique", "size (MB)"],
        )
    )
    print("Total size:", df.memory_usage(deep=True).sum() / 1024**2, "MB")
    print(f"df.shape:' {df.shape}")


df = pd.read_csv(
    "datasets/Motor_Vehicle_Collisions_-_Crashes_20260407.csv", delimiter=","
)

summarize_columns(df)

/tmp/ipykernel_7978/719224584.py:23: DtypeWarning: Columns (0: ZIP CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


                             name    dtype   unique  size (MB)
0                      CRASH DATE      str     5025         38
1                      CRASH TIME      str     1440         27
2                         BOROUGH      str        6         28
3                        ZIP CODE   object      439         77
4                        LATITUDE  float64   130328         17
5                       LONGITUDE  float64   101116         17
6                        LOCATION      str   393219         62
7                  ON STREET NAME      str    23176         65
8               CROSS STREET NAME      str    25238         46
9                 OFF STREET NAME      str   267508         30
10      NUMBER OF PERSONS INJURED  float64       33         17
11       NUMBER OF PERSONS KILLED  float64        8         17
12  NUMBER OF PEDESTRIANS INJURED    int64       14         17
13   NUMBER OF PEDESTRIANS KILLED    int64        6         17
14      NUMBER OF CYCLIST INJURED    int64        5    

## Discarding invalid rows & Redundant Columns

In [125]:
# --- Removing Lat/Lon NaN values ---#

rows_before = df.shape[0]
df = df[df["LONGITUDE"].notna() & df["LATITUDE"].notna()]
print(f"-> Dropped {rows_before - df.shape[0]} rows after removing lat/lon NaNs rows")

-> Dropped 240676 rows after removing lat/lon NaNs rows


In [126]:
# --- Removing redundant columns ---#

df = df.drop(["ZIP CODE", "LOCATION", "COLLISION_ID"], axis=1)

In [127]:
summarize_columns(df)

                             name    dtype  unique  size (MB)
0                      CRASH DATE      str    5025         50
1                      CRASH TIME      str    1440         40
2                         BOROUGH      str       6         41
3                        LATITUDE  float64  130327         30
4                       LONGITUDE  float64  101115         30
5                  ON STREET NAME      str   16812         73
6               CROSS STREET NAME      str   19041         56
7                 OFF STREET NAME      str  250087         43
8       NUMBER OF PERSONS INJURED  float64      31         30
9        NUMBER OF PERSONS KILLED  float64       8         30
10  NUMBER OF PEDESTRIANS INJURED    int64      14         30
11   NUMBER OF PEDESTRIANS KILLED    int64       6         30
12      NUMBER OF CYCLIST INJURED    int64       5         30
13       NUMBER OF CYCLIST KILLED    int64       3         30
14     NUMBER OF MOTORIST INJURED    int64      29         30
15      

## Fixing missing BUROUGHS

In [128]:
import geopandas as gpd
from geodatasets import get_path

# -- Title Entries ex: BRONX -> Bronx
df["BOROUGH"] = df["BOROUGH"].str.title()


print(
    "Unique Boroughs include",
    list(df["BOROUGH"].unique()),
    "\nNumber of rows with missing 'BOROUGH' but present coordinates is:",
    sum(df["BOROUGH"].isna()),
)

missing_mask = df["BOROUGH"].isna()

missing = df.loc[missing_mask, ["LONGITUDE", "LATITUDE"]]

gdf_missing = gpd.GeoDataFrame(
    missing,
    geometry=gpd.points_from_xy(missing["LONGITUDE"], missing["LATITUDE"]),
    crs="EPSG:4326",
)

# --- Using geodatasets to source boroughs multipoligon ---#
path_to_boroughs = get_path("nybb")
boroughs = gpd.read_file(path_to_boroughs)

boroughs = boroughs.to_crs(
    "EPSG:4326"
)  # Convert to standard coordinate reference system


# --- Left spacial join gdf_missing with boroughs multi-poligon ---#
joined = gpd.sjoin(
    gdf_missing, boroughs[["BoroName", "geometry"]], how="left", predicate="within"
)

df.loc[missing_mask, "BOROUGH"] = joined["BoroName"]

print("Remaining missing BOROUGH after spatial fill:", df["BOROUGH"].isna().sum())

# --- Removing rows with BOROUGH == nan, after the fill ---#

df = df[df["BOROUGH"].notna()]
summarize_columns(df)

Unique Boroughs include ['Brooklyn', nan, 'Bronx', 'Manhattan', 'Queens', 'Staten Island'] 
Number of rows with missing 'BOROUGH' but present coordinates is: 484641
Remaining missing BOROUGH after spatial fill: 10094
                             name    dtype  unique  size (MB)
0                      CRASH DATE      str    5025         49
1                      CRASH TIME      str    1440         39
2                         BOROUGH      str       5         44
3                        LATITUDE  float64  130276         30
4                       LONGITUDE  float64  101069         30
5                  ON STREET NAME      str   16773         72
6               CROSS STREET NAME      str   19035         56
7                 OFF STREET NAME      str  249980         42
8       NUMBER OF PERSONS INJURED  float64      31         30
9        NUMBER OF PERSONS KILLED  float64       8         30
10  NUMBER OF PEDESTRIANS INJURED    int64      14         30
11   NUMBER OF PEDESTRIANS KILLED    in

,index,CRASH DATE,CRASH TIME,BOROUGH,LATITUDE,LONGITUDE,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,NUMBER OF PERSONS INJURED,...,CONTRIBUTING FACTOR VEHICLE 1,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,2,11/01/2023,1:29,Brooklyn,40.621790,-73.970024,OCEAN PARKWAY,AVENUE K,NaN,1.0,...,Unspecified,Unspecified,Unspecified,NaN,NaN,Moped,Sedan,Sedan,NaN,NaN
1,9,09/11/2021,9:35,Brooklyn,40.667202,-73.866500,NaN,NaN,1211 LORING AVENUE,0.0,...,Unspecified,NaN,NaN,NaN,NaN,Sedan,NaN,NaN,NaN,NaN
2,10,12/14/2021,8:13,Brooklyn,40.683304,-73.917274,SARATOGA AVENUE,DECATUR STREET,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,12,12/14/2021,17:05,Brooklyn,40.709183,-73.956825,BROOKLYN QUEENS EXPRESSWAY,NaN,NaN,0.0,...,Passing Too Closely,Unspecified,NaN,NaN,NaN,Sedan,Tractor Truck Diesel,NaN,NaN,NaN
4,13,12/14/2021,8:17,Bronx,40.868160,-73.831480,NaN,NaN,344 BAYCHESTER AVENUE,2.0,...,Unspecified,Unspecified,NaN,NaN,NaN,Sedan,Sedan,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2002417,2253186,04/20/2021,20:20,Staten Island,40.622635,-74.160810,NaN,NaN,105 LUDWIG LANE,0.0,...,Driver Inattention/Distraction,Unspecified,NaN,NaN,NaN,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN
2002418,2253188,04/20/2021,15:07,Brooklyn,40.683480,-73.966896,FULTON STREET,NaN,NaN,1.0,...,Unspecified,Unspecified,NaN,NaN,NaN,Convertible,Moped,NaN,NaN,NaN
2002419,2253189,04/20/2021,17:00,Bronx,40.855600,-73.885150,NaN,NaN,660 EAST 188 STREET,0.0,...,Passing or Lane Usage Improper,Unspecified,NaN,NaN,NaN,Sedan,NaN,NaN,NaN,NaN
2002420,2253190,04/20/2021,16:45,Brooklyn,40.625828,-73.990950,NaN,NaN,5822 16 AVENUE,0.0,...,Unspecified,NaN,NaN,NaN,NaN,Sedan,NaN,NaN,NaN,NaN


## Memory Optimization

In [94]:
df["CRASH DATETIME"] = pd.to_datetime(df["CRASH DATE"] + " " + df["CRASH TIME"], format= "%m/%d/%Y %H:%M" )

df = df.drop(["CRASH TIME", "CRASH DATE"], axis=1)

summarize_columns(df)
print(df.columns)

                             name           dtype   unique  size (MB)
0                           index           int64  2002422         15
1                         BOROUGH             str        5         29
2                        LATITUDE         float64   130276         15
3                       LONGITUDE         float64   101069         15
4                  ON STREET NAME             str    16773         57
5               CROSS STREET NAME             str    19035         41
6                 OFF STREET NAME             str   249980         27
7       NUMBER OF PERSONS INJURED         float64       31         15
8        NUMBER OF PERSONS KILLED         float64        8         15
9   NUMBER OF PEDESTRIANS INJURED           int64       14         15
10   NUMBER OF PEDESTRIANS KILLED           int64        6         15
11      NUMBER OF CYCLIST INJURED           int64        5         15
12       NUMBER OF CYCLIST KILLED           int64        3         15
13     NUMBER OF MOT

In [ ]:
df

,index,BOROUGH,LATITUDE,LONGITUDE,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5,CRASH DATETIME
0,2,Brooklyn,40.621790,-73.970024,OCEAN PARKWAY,AVENUE K,NaN,1.0,0.0,0,...,Unspecified,Unspecified,NaN,NaN,Moped,Sedan,Sedan,NaN,NaN,2023-11-01 01:29:00
1,9,Brooklyn,40.667202,-73.866500,NaN,NaN,1211 LORING AVENUE,0.0,0.0,0,...,NaN,NaN,NaN,NaN,Sedan,NaN,NaN,NaN,NaN,2021-09-11 09:35:00
2,10,Brooklyn,40.683304,-73.917274,SARATOGA AVENUE,DECATUR STREET,NaN,0.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-12-14 08:13:00
3,12,Brooklyn,40.709183,-73.956825,BROOKLYN QUEENS EXPRESSWAY,NaN,NaN,0.0,0.0,0,...,Unspecified,NaN,NaN,NaN,Sedan,Tractor Truck Diesel,NaN,NaN,NaN,2021-12-14 17:05:00
4,13,Bronx,40.868160,-73.831480,NaN,NaN,344 BAYCHESTER AVENUE,2.0,0.0,0,...,Unspecified,NaN,NaN,NaN,Sedan,Sedan,NaN,NaN,NaN,2021-12-14 08:17:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2002417,2253186,Staten Island,40.622635,-74.160810,NaN,NaN,105 LUDWIG LANE,0.0,0.0,0,...,Unspecified,NaN,NaN,NaN,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN,2021-04-20 20:20:00
2002418,2253188,Brooklyn,40.683480,-73.966896,FULTON STREET,NaN,NaN,1.0,0.0,0,...,Unspecified,NaN,NaN,NaN,Convertible,Moped,NaN,NaN,NaN,2021-04-20 15:07:00
2002419,2253189,Bronx,40.855600,-73.885150,NaN,NaN,660 EAST 188 STREET,0.0,0.0,0,...,Unspecified,NaN,NaN,NaN,Sedan,NaN,NaN,NaN,NaN,2021-04-20 17:00:00
2002420,2253190,Brooklyn,40.625828,-73.990950,NaN,NaN,5822 16 AVENUE,0.0,0.0,0,...,NaN,NaN,NaN,NaN,Sedan,NaN,NaN,NaN,NaN,2021-04-20 16:45:00


In [ ]:
df = df.astype(
    {
        
    }
)